# A1 — Profiler un relevé:

In [2]:
import pandas as pd

# Charger un relevé (un seul fichier)
df = pd.read_json("releves/chronovet/2026-04-01_1919/products.jsonl", lines=True)

print("Nombre total de lignes :", len(df))

# Lignes sans EAN
df_no_ean = df[df["ean"].isna() | (df["ean"] == "")]   # nb des valeurs ean manquantes ou vides 
print("Nombre de lignes sans EAN :", len(df_no_ean))

# Doublons d'EAN
dup_ean = df["ean"].value_counts()
n_ean_duplicates = (dup_ean > 1).sum()
print("Nombre d’EAN dupliqués :", n_ean_duplicates)

# Colonnes présentes
print("Colonnes :", list(df.columns))


Nombre total de lignes : 1631
Nombre de lignes sans EAN : 0
Nombre d’EAN dupliqués : 0
Colonnes : ['site', 'url', 'scraped_at', 'ean', 'sku', 'site_id', 'name', 'brand', 'category', 'price', 'currency', 'in_stock', 'image_url', 'extra']


# A3 — Charger les relevés du site

In [4]:
import os
import json
import pandas as pd
import psycopg2
import unicodedata

# -----------------------------
# Connexion PostgreSQL
# -----------------------------
conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
cur = conn.cursor()

 #utiliser pour identifier les lignes (i.e. s'il y a un doublon sur (clef, site, scraped_at) 
 #avec clef = ean si présent, sinon url
# -----------------------------
# Fonction : calculer la cle
# -----------------------------
def compute_cle(row):
    ean = row.get("ean")
    if isinstance(ean, float) or ean in [None, ""]:
        return row.get("url")
    return str(ean)

# -----------------------------
# Fonction : insérer / trouver produit
# -----------------------------
def upsert_product(row):
    cle = compute_cle(row)

    # 1. Chercher produit existant

        #Si un produit avec cette cle existe → je le réutilises
        #Rejouabilité garantie  
        #Pas de doublons dans product

    cur.execute("SELECT product_id FROM product WHERE cle = %s", (cle,))
    res = cur.fetchone()
    if res:
        return res[0]

    # 2. Insérer produit
    cur.execute("""
        INSERT INTO product (cle, ean, name, brand, category, description_short,
                             variant_name, pack_size, species, conditioning,
                             weight_kg, mpn, atc_code, image_url, extra)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        RETURNING product_id
    """, (
        cle,
        row.get("ean"),
        row.get("name"),
        row.get("brand"),
        row.get("category"),
        row.get("description_short"),
        row.get("variant_name"),
        row.get("pack_size"),
        row.get("species"),
        row.get("conditioning"),
        row.get("weight_kg"),
        row.get("mpn"),
        row.get("atc_code"),
        row.get("image_url"),
        json.dumps(row.get("extra"))
    ))
    return cur.fetchone()[0]

# -----------------------------
# Fonction : insérer relevé
# -----------------------------
def insert_price_fact(row, product_id):
    cle = compute_cle(row)

    cur.execute("""
        INSERT INTO price_fact (product_id, cle, site, url, scraped_at,
                                price, price_was, currency, in_stock,
                                stock_text, price_per_unit, price_per_unit_label,
                                rating, review_count, extra)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        ON CONFLICT DO NOTHING
    """, (
        product_id,
        cle,
        row.get("site"),
        row.get("url"),
        row.get("scraped_at"),
        row.get("price"),
        row.get("price_was"),
        row.get("currency"),
        row.get("in_stock"),
        row.get("stock_text"),
        row.get("price_per_unit"),
        row.get("price_per_unit_label"),
        row.get("rating"),
        row.get("review_count"),
        json.dumps(row.get("extra"))
    ))

# -----------------------------
# Chargement d'un relevé (A3)
# -----------------------------
def load_one_releve(path):
    print(f"Chargement d'un relevé : {path}")

    df = pd.read_json(path, lines=True)

    # Calcul de cle
    df["cle"] = df.apply(compute_cle, axis=1)

    # Dédoublonnage interne (un seul relevé)
    df = df.drop_duplicates(subset=["cle"], keep="first")

    for _, row in df.iterrows():
        row = row.to_dict()
        product_id = upsert_product(row)
        insert_price_fact(row, product_id)

    conn.commit()
    print("OK")

# -----------------------------
# Chargement multi-dates (A3)
# -----------------------------
def load_multi_dates(paths):
    print("Chargement multi-dates...")

    # Rejouabilité : vider price_fact
    cur.execute("TRUNCATE price_fact;")
    conn.commit()

    dfs = []
    for p in paths:
        df = pd.read_json(p, lines=True)
        df["cle"] = df.apply(compute_cle, axis=1)
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)

    # Dédoublonnage multi-dates
    df_all = df_all.drop_duplicates(subset=["cle", "scraped_at"], keep="first")

    for _, row in df_all.iterrows():
        row = row.to_dict()
        product_id = upsert_product(row)
        insert_price_fact(row, product_id)

    conn.commit()
    print("Chargement multi-dates terminé.")

# -----------------------------
# Exécution A3
# -----------------------------
load_one_releve("releves/chronovet/2026-04-01_1919/products.jsonl")

load_multi_dates([
    "releves/chronovet/2026-07-12_1150/products.jsonl",
    "releves/chronovet/2026-07-12_1242/products.jsonl",
    "releves/chronovet/2026-07-18_1334/products.jsonl"
])

print("Pipeline A3 terminé.")


Chargement d'un relevé : releves/chronovet/2026-04-01_1919/products.jsonl
OK
Chargement multi-dates...
Chargement multi-dates terminé.
Pipeline A3 terminé.
